# Detecção de capacete e colete (EPI) com YOLO
EDA, pré-processamento, treino de 3 modelos, tuning e avaliação no teste.

## 1. Setup

In [ ]:
!pip -q install -U ultralytics

In [ ]:
import os, glob, shutil, zipfile, hashlib, random, yaml
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch
from IPython.display import display
from ultralytics import YOLO

os.environ['WANDB_MODE'] = 'disabled'
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SEM GPU')

In [ ]:
BASE = Path('/kaggle/working')
RUNS = BASE / 'runs'
NAMES = ['helmet', 'no_helmet', 'no_vest', 'person', 'vest']
SPLITS = ['train', 'valid', 'test']

DS = next(Path('/kaggle/input').rglob('data.yaml')).parent
print(DS)

## 2. EDA

In [ ]:
def carregar_labels(base):
    linhas = []
    for s in SPLITS:
        for f in (base / s / 'labels').glob('*.txt'):
            for l in f.read_text().splitlines():
                v = l.split()
                if len(v) == 5:
                    linhas.append((s, f.stem, int(v[0]), *map(float, v[1:])))
    df = pd.DataFrame(linhas, columns=['split', 'img', 'cls', 'x', 'y', 'w', 'h'])
    df['classe'] = df.cls.map(dict(enumerate(NAMES)))
    df['area'] = df.w * df.h
    return df

df = carregar_labels(DS)
n_imgs = pd.Series({s: len(list((DS / s / 'images').glob('*'))) for s in SPLITS}, name='imagens')
tab = pd.crosstab(df.classe, df.split)[SPLITS]

display(n_imgs.to_frame())
display(tab)
tab.plot.bar(figsize=(8, 4), title='Instâncias por classe e split')
plt.show()

In [ ]:
por_img = df.groupby(['split', 'img']).size()
sem_anotacao = sum(n_imgs[s] - len(por_img[s]) for s in SPLITS)
print('imagens sem anotação:', sem_anotacao)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
por_img.hist(bins=range(1, 31), ax=ax[0])
ax[0].set_title('Caixas por imagem')
df.boxplot(column='area', by='classe', ax=ax[1])
ax[1].set_yscale('log')
ax[1].set_title('Área relativa da caixa')
plt.suptitle('')
plt.show()

print('proporção de caixas pequenas (área < 1% da imagem):')
print(df.assign(pequeno=df.area < 0.01).groupby('classe').pequeno.mean().round(3))

In [ ]:
COR = dict(zip(NAMES, [(0, 200, 0), (220, 0, 0), (255, 140, 0), (0, 120, 255), (160, 0, 200)]))

def desenhar(img_p, lab_p):
    img = cv2.cvtColor(cv2.imread(str(img_p)), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    linhas = lab_p.read_text().splitlines() if lab_p.exists() else []
    for l in linhas:
        v = l.split()
        if len(v) != 5:
            continue
        nome = NAMES[int(v[0])]
        x, y, w, h = map(float, v[1:])
        x1, y1 = int((x - w / 2) * W), int((y - h / 2) * H)
        x2, y2 = int((x + w / 2) * W), int((y + h / 2) * H)
        cv2.rectangle(img, (x1, y1), (x2, y2), COR[nome], 2)
        cv2.putText(img, nome, (x1, max(y1 - 4, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, COR[nome], 1)
    return img

random.seed(1)
amostra = random.sample(list((DS / 'train' / 'images').glob('*')), 8)
fig, ax = plt.subplots(2, 4, figsize=(16, 8))
for a, p in zip(ax.ravel(), amostra):
    a.imshow(desenhar(p, DS / 'train' / 'labels' / (p.stem + '.txt')))
    a.axis('off')
plt.tight_layout()
plt.show()

## 3. Pré-processamento
Remove duplicatas (dentro e entre splits, priorizando test e valid) e linhas de anotação inválidas.

In [ ]:
def md5(p):
    return hashlib.md5(p.read_bytes()).hexdigest()

vistos = set()
manter = {}
removidas = {}
for s in ['test', 'valid', 'train']:
    manter[s] = []
    for p in sorted((DS / s / 'images').glob('*')):
        h = md5(p)
        if h not in vistos:
            vistos.add(h)
            manter[s].append(p)
    removidas[s] = n_imgs[s] - len(manter[s])

print('duplicatas removidas:', removidas)

In [ ]:
CLEAN = Path('/kaggle/temp/ppe_clean')
shutil.rmtree(CLEAN, ignore_errors=True)
linhas_invalidas = 0

for s in SPLITS:
    (CLEAN / s / 'images').mkdir(parents=True)
    (CLEAN / s / 'labels').mkdir(parents=True)
    for p in manter[s]:
        shutil.copy(p, CLEAN / s / 'images' / p.name)
        lab = DS / s / 'labels' / (p.stem + '.txt')
        ok = []
        if lab.exists():
            for l in lab.read_text().splitlines():
                if not l.strip():
                    continue
                v = l.split()
                valida = len(v) == 5 and int(v[0]) in range(len(NAMES)) and all(0 <= float(x) <= 1 for x in v[1:])
                if valida:
                    ok.append(l)
                else:
                    linhas_invalidas += 1
        (CLEAN / s / 'labels' / (p.stem + '.txt')).write_text('\n'.join(ok))

yaml.safe_dump(
    {'path': str(CLEAN), 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images',
     'nc': len(NAMES), 'names': NAMES},
    open(CLEAN / 'data.yaml', 'w'))

DATA = str(CLEAN / 'data.yaml')
print('linhas inválidas removidas:', linhas_invalidas)
print({s: len(list((CLEAN / s / 'images').glob('*'))) for s in SPLITS})

## 4. Treino: comparação de modelos
YOLO26n, YOLO26s e YOLO11s com os mesmos parâmetros. A escolha é feita no conjunto de validação.

In [ ]:
MODELOS = ['yolo26n.pt', 'yolo26s.pt', 'yolo11s.pt']
EPOCHS = 50
linhas = []
pesos = {}

for m in MODELOS:
    nome = m.replace('.pt', '')
    modelo = YOLO(m)
    modelo.train(data=DATA, epochs=EPOCHS, imgsz=640, batch=16, patience=15, seed=42,
                 project=str(RUNS), name='base_' + nome, exist_ok=True)
    pesos[nome] = modelo.trainer.best
    r = YOLO(pesos[nome]).val(data=DATA, split='val', imgsz=640, plots=False)
    linhas.append({'modelo': nome, 'precisao': r.box.mp, 'recall': r.box.mr,
                   'mAP50': r.box.map50, 'mAP50-95': r.box.map, 'ms_por_imagem': r.speed['inference']})

tabela = pd.DataFrame(linhas).sort_values('mAP50-95', ascending=False).reset_index(drop=True)
display(tabela.round(4))
tabela.set_index('modelo')[['mAP50', 'mAP50-95']].plot.bar(figsize=(7, 4), ylim=(0, 1))
plt.show()

## 5. Tuning de hiperparâmetros
Busca evolutiva no melhor modelo. Aumente `TUNE_ITER` se houver tempo de GPU.

In [ ]:
TUNE_ITER = 10
TUNE_EPOCHS = 15
melhor = tabela.loc[0, 'modelo']
print('modelo escolhido:', melhor)

YOLO(melhor + '.pt').tune(data=DATA, epochs=TUNE_EPOCHS, iterations=TUNE_ITER, optimizer='AdamW',
                          imgsz=640, batch=16, plots=False, save=False, val=False,
                          project=str(RUNS), name='tune', exist_ok=True)

In [ ]:
achados = glob.glob('/kaggle/working/**/best_hyperparameters.yaml', recursive=True)
hp_file = max(achados, key=os.path.getmtime)
hp = yaml.safe_load(open(hp_file))
print(hp_file)
print(hp)

## 6. Treino final com os hiperparâmetros otimizados

In [ ]:
FINAL_EPOCHS = 80
final = YOLO(melhor + '.pt')
final.train(data=DATA, epochs=FINAL_EPOCHS, imgsz=640, batch=16, patience=20, seed=42,
            project=str(RUNS), name='final', exist_ok=True, **hp)
PESO_FINAL = final.trainer.best
print(PESO_FINAL)

## 7. Avaliação no conjunto de teste

In [ ]:
def avaliar(peso, tag, plots):
    r = YOLO(peso).val(data=DATA, split='test', imgsz=640, plots=plots,
                       project=str(RUNS), name='teste_' + tag, exist_ok=True)
    geral = {'modelo': tag, 'precisao': r.box.mp, 'recall': r.box.mr,
             'mAP50': r.box.map50, 'mAP50-95': r.box.map, 'ms_por_imagem': r.speed['inference']}
    por_classe = pd.DataFrame({'classe': [r.names[i] for i in r.box.ap_class_index],
                               'precisao': r.box.p, 'recall': r.box.r,
                               'mAP50': r.box.ap50, 'mAP50-95': r.box.ap})
    return geral, por_classe, r

g_base, _, _ = avaliar(pesos[melhor], 'baseline_' + melhor, False)
g_final, por_classe, r_final = avaliar(PESO_FINAL, 'final_tuned', True)

display(pd.DataFrame([g_base, g_final]).round(4))
display(por_classe.round(4))

In [ ]:
for nome in ['confusion_matrix_normalized.png', 'confusion_matrix.png']:
    f = Path(r_final.save_dir) / nome
    if f.exists():
        plt.figure(figsize=(8, 7))
        plt.imshow(plt.imread(f))
        plt.axis('off')
        plt.show()
        break

In [ ]:
modelo_final = YOLO(PESO_FINAL)
random.seed(7)
teste = random.sample(list((CLEAN / 'test' / 'images').glob('*')), 8)
fig, ax = plt.subplots(2, 4, figsize=(16, 8))
for a, p in zip(ax.ravel(), teste):
    res = modelo_final.predict(str(p), conf=0.25, verbose=False)[0]
    a.imshow(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB))
    a.axis('off')
plt.tight_layout()
plt.show()

## 8. Salvar artefatos

In [ ]:
shutil.copy(PESO_FINAL, BASE / 'best_epi.pt')
shutil.copy(CLEAN / 'data.yaml', BASE / 'data_epi.yaml')
pd.DataFrame([g_base, g_final]).to_csv(BASE / 'metricas_teste.csv', index=False)
por_classe.to_csv(BASE / 'metricas_por_classe_teste.csv', index=False)
print('salvo em', BASE)

In [ ]:
from pathlib import Path
import shutil
from IPython.display import FileLink

B = Path('/kaggle/working')
E = B / 'entrega'
E.mkdir(exist_ok=True)
for n in ['best_epi.pt', 'data_epi.yaml', 'metricas_teste.csv', 'metricas_por_classe_teste.csv']:
    shutil.copy(B / n, E / n)
shutil.make_archive(str(B / 'epi_resultados'), 'zip', E)
FileLink('epi_resultados.zip')

In [ ]:
!pip -q install imagehash

In [ ]:
import imagehash
from PIL import Image

def hashes(split):
    return {p.name: imagehash.phash(Image.open(p).convert('RGB'), hash_size=8)
            for p in sorted((CLEAN / split / 'images').glob('*'))}

def matriz(h):
    nomes = list(h)
    arr = np.array([h[n].hash.flatten() for n in nomes], dtype=np.uint8)
    return nomes, arr

def distancias(a, b):
    # distância de Hamming entre todos os pares (0 a 64)
    return (a[:, None, :] != b[None, :, :]).sum(-1)

H = {s: hashes(s) for s in SPLITS}
n_te, a_te = matriz(H['test'])
n_va, a_va = matriz(H['valid'])
n_tr, a_tr = matriz(H['train'])

d_te_tr = distancias(a_te, a_tr); min_te_tr = d_te_tr.min(1); viz_te_tr = d_te_tr.argmin(1)
d_va_tr = distancias(a_va, a_tr); min_va_tr = d_va_tr.min(1)
d_te_va = distancias(a_te, a_va); min_te_va = d_te_va.min(1)

LIMIAR = 5  # ponto de partida comum; confira nos pares visuais abaixo
for nome, d in [('teste vs treino', min_te_tr), ('valid vs treino', min_va_tr), ('teste vs valid', min_te_va)]:
    cont = {f'<= {t}': int((d <= t).sum()) for t in (0, 2, 5, 8, 10)}
    print(f'{nome} (de {len(d)} imagens):', cont)

# 3 pares mais próximos e 3 pares na faixa de distância 8 (borderline)
ordem = list(np.argsort(min_te_tr)[:3]) + list(np.argsort(np.abs(min_te_tr - 8))[:3])
fig, ax = plt.subplots(len(ordem), 2, figsize=(9, 3.2 * len(ordem)))
for r, i in enumerate(ordem):
    ax[r, 0].imshow(Image.open(CLEAN / 'test' / 'images' / n_te[i]))
    ax[r, 0].set_title(f'teste (distância {min_te_tr[i]})')
    ax[r, 1].imshow(Image.open(CLEAN / 'train' / 'images' / n_tr[viz_te_tr[i]]))
    ax[r, 1].set_title('treino mais parecido')
    ax[r, 0].axis('off'); ax[r, 1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
PESO = PESO_FINAL if 'PESO_FINAL' in globals() else '/kaggle/working/best_epi.pt'
dup = {n_te[i] for i in np.where(min_te_tr <= LIMIAR)[0]}
print('imagens de teste removidas:', len(dup), 'de', len(n_te))

LIMPO = Path('/kaggle/temp/ppe_test_sem_dup')
shutil.rmtree(LIMPO, ignore_errors=True)
(LIMPO / 'test' / 'images').mkdir(parents=True)
(LIMPO / 'test' / 'labels').mkdir(parents=True)
for p in (CLEAN / 'test' / 'images').glob('*'):
    if p.name in dup:
        continue
    shutil.copy(p, LIMPO / 'test' / 'images' / p.name)
    lab = CLEAN / 'test' / 'labels' / (p.stem + '.txt')
    if lab.exists():
        shutil.copy(lab, LIMPO / 'test' / 'labels' / lab.name)

yaml.safe_dump({'path': str(LIMPO), 'train': 'test/images', 'val': 'test/images',
                'test': 'test/images', 'nc': len(NAMES), 'names': NAMES},
               open(LIMPO / 'data.yaml', 'w'))

m = YOLO(PESO)
r_todo = m.val(data=DATA, split='test', imgsz=640, plots=False, verbose=False)
r_limpo = m.val(data=str(LIMPO / 'data.yaml'), split='test', imgsz=640, plots=False, verbose=False)

comp = pd.DataFrame({
    'teste completo': [r_todo.box.map50, r_todo.box.map, r_todo.box.mr],
    'teste sem quase-duplicatas': [r_limpo.box.map50, r_limpo.box.map, r_limpo.box.mr],
}, index=['mAP50', 'mAP50-95', 'recall']).round(4)
display(comp)